Benchmarking Standard

- Select Dataset and Store it on Google Drive v2

In [ ]:
!pip install datasets pillow google-generativeai requests pandas tqdm --quiet

In [ ]:
# ============================================
# 0. SETUP
# ============================================
from google.colab import drive
drive.mount('/content/drive')


import os, json, time, requests, ast, re, unicodedata
import pandas as pd
from tqdm import tqdm
from PIL import Image
from io import BytesIO
from datasets import load_dataset
import google.generativeai as genai
import google.api_core.exceptions as google_exceptions
import time
import multiprocessing as mp
import numpy as np
from google.colab import userdata

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# ============================================
# 1. LOAD MM-Food-100K DATASET
# ============================================
ds = load_dataset("Codatta/MM-Food-100K")
train = ds["train"]

# Define food groups
processed_types = ["Restaurant food", "Home-cooked food", "Processed food", "Snack"]
simple_types = ["Raw", "Fruit", "Vegetable"]

# Filter into processed vs. simple subsets
processed_foods = train.filter(lambda x: x["food_type"] in processed_types)
simple_foods    = train.filter(lambda x: x["food_type"] in simple_types)

print("Processed foods:", len(processed_foods))
print("Simple foods:", len(simple_foods))

processed_subset = processed_foods.select(range(400))

# ============================================
# 2. LOAD SIMPLE FRUIT/VEG DATASET
# ============================================
simple_ds = load_dataset("Nattakarn/fruit-and-vegetable-image-recognition")
label_names = simple_ds["train"].features["label"].names


# ============================================
# 3. DOWNLOAD PROCESSED FOOD IMAGES
# ============================================
os.makedirs("processed_images", exist_ok=True)

processed_rows = []
saved = 0
# Change target to adjust total sample image number
target = 400
# target = 5
i = 0

while saved < target and i < len(processed_foods):
    row = processed_foods[i]
    url = row["image_url"]

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))

        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")

        filename = f"processed_images/processed_{saved}.jpg"

        img = img.resize((384, 384))
        img.save(
            filename,
            format="JPEG",
            quality=85,
            optimize=False,
            progressive=False
        )


        processed_rows.append({
            "image_path": filename,
            "label": row["dish_name"],
            "food_type": "processed",
            "cooking_method": row["cooking_method"],
            "portion_size": row["portion_size"],
            "calories_kcal": row["nutritional_profile"],
        })

        saved += 1

    except Exception as e:
        print(f"Skipping {url}: {e}")

    i += 1


print(f"Downloaded {saved} processed images.")


# ============================================
# 4. DOWNLOAD SIMPLE FRUIT/VEG IMAGES
# ============================================
os.makedirs("simple_images", exist_ok=True)

simple_rows = []
saved = 0
# Change target to adjust total sample image number
target = 100
# target = 5
i = 0

while saved < target and i < len(simple_ds["train"]):
    item = simple_ds["train"][i]
    label_name = label_names[item["label"]]

    try:
        img = item["image"]

        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")

        filename = f"simple_images/simple_{saved}.jpg"

        img = img.resize((384, 384))
        img.save(
            filename,
            format="JPEG",
            quality=85,
            optimize=False,
            progressive=False
        )


        simple_rows.append({
            "image_path": filename,
            "label": label_name,
            "food_type": "simple",
            "cooking_method": "raw",
            "portion_size": None,
            "calories_kcal": None,
        })

        saved += 1

    except Exception as e:
        print(f"Skipping index {i}: {e}")

    i += 1


print(f"Downloaded {saved} simple images.")


# ============================================
# 5. SAVE DATASET CSV
# ============================================
import csv

all_rows = processed_rows + simple_rows

csv_path = "benchmark_prototype_dataset.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    # CSV header
    writer.writerow([
        "image_path",
        "label",
        "food_type",
        "cooking_method",
        "portion_size",
        "calories_kcal"
    ])

    # Write each benchmark row
    for row in all_rows:
        writer.writerow([
            row["image_path"],
            row["label"],
            row["food_type"],
            row["cooking_method"],
            row["portion_size"],
            row["calories_kcal"]
        ])

print("CSV saved:", csv_path)

# Move files to Google Drive
!mkdir -p /content/drive/MyDrive/FoodBenchmarkDataset
!cp -r simple_images processed_images benchmark_prototype_dataset.csv /content/drive/MyDrive/FoodBenchmarkDataset/





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

MM-Food-100K.csv:   0%|          | 0.00/28.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100000 [00:00<?, ? examples/s]

...

data/train-00002-of-00003.parquet:   0%|          | 0.00/470M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/292M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2492 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/623 [00:00<?, ? examples/s]

Downloaded 400 processed images.
Downloaded 100 simple images.
CSV saved: benchmark_prototype_dataset.csv


In [ ]:
# ============================================
# DRIVE SETUP
# ============================================

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


- Load the Stored Dataset and Call Gemini API

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
BASE = "/content/drive/MyDrive/FoodBenchmarkDataset/"
df = pd.read_csv(BASE + "benchmark_prototype_dataset.csv")
df["image_path"] = df["image_path"].apply(lambda p: os.path.join(BASE, p))

simple_df    = df[df["food_type"] == "simple"].reset_index(drop=True)
processed_df = df[df["food_type"] == "processed"].reset_index(drop=True)


####################### V2
# Limit samples for fast testing. If you don't need this, you can comment out all these three lines.
# TEST_N = 5
# simple_df = simple_df.sample(n=TEST_N, random_state=42).reset_index(drop=True)
# processed_df = processed_df.sample(n=TEST_N, random_state=42).reset_index(drop=True)


print("Simple:", len(simple_df))
print("Processed:", len(processed_df))

# ============================================================
# 1. GEMINI SETUP
# ============================================================
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-2.0-flash-lite")



# before aksing model to answer it , give few examples to the model to increase precision of return value
PROMPT = """
You are a food analysis model.
Your job is to analyze a food image and output STRICT valid JSON.

Below are EXAMPLES of perfect JSON. Follow them EXACTLY.

=== EXAMPLE 1 (simple raw food) ===
{
  "dish_name": "apple",
  "ingredients": ["apple"],
  "portion_size": ["apple:150g"],
  "cooking_method": "raw",
  "nutritional_profile": {
    "calories_kcal": 95,
    "protein_g": 0.5,
    "fat_g": 0.3,
    "carbohydrate_g": 25
  }
}

=== EXAMPLE 2 (simple vegetable) ===
{
  "dish_name": "broccoli",
  "ingredients": ["broccoli"],
  "portion_size": ["broccoli:90g"],
  "cooking_method": "raw",
  "nutritional_profile": {
    "calories_kcal": 31,
    "protein_g": 2.5,
    "fat_g": 0.3,
    "carbohydrate_g": 6
  }
}

=== EXAMPLE 3 (processed/homemade meal) ===
{
  "dish_name": "fried chicken",
  "ingredients": ["chicken", "oil", "flour"],
  "portion_size": ["chicken:200g"],
  "cooking_method": "fried",
  "nutritional_profile": {
    "calories_kcal": 450,
    "protein_g": 28,
    "fat_g": 25,
    "carbohydrate_g": 22
  }
}

=== EXAMPLE 4 (restaurant dish) ===
{
  "dish_name": "spaghetti bolognese",
  "ingredients": ["spaghetti", "beef", "tomato", "onion"],
  "portion_size": ["spaghetti:200g", "beef:100g"],
  "cooking_method": "boiled and simmered",
  "nutritional_profile": {
    "calories_kcal": 680,
    "protein_g": 32,
    "fat_g": 20,
    "carbohydrate_g": 90
  }
}

Now analyze the food image and return STRICT JSON ONLY:
{
  "dish_name": "",
  "ingredients": [],
  "portion_size": [],
  "cooking_method": "",
  "nutritional_profile": {
    "calories_kcal": 0,
    "protein_g": 0,
    "fat_g": 0,
    "carbohydrate_g": 0
  }
}
"""

# ============================================================
# 2. HELPERS
# ============================================================
def load_image(path):
    try:
        img = Image.open(path)
        if img.mode in ("RGBA","P"): img = img.convert("RGB")
        return img
    except:
        return None

def extract_json(text):
    if not text: return None
    start = text.find("{")
    if start < 0: return None
    depth = 0
    for i,ch in enumerate(text[start:], start=start):
        if ch=="{": depth+=1
        if ch=="}":
            depth-=1
            if depth==0:
                try: return json.loads(text[start:i+1])
                except: return None
    return None

# Worker for timeout-safe call
def worker(img, ret):
    try:
        resp = model.generate_content([PROMPT, img], generation_config={"temperature": 0})
        ret["out"] = resp
    except Exception as e:
        ret["error"] = str(e)


last_call_time = None



#### V2
def safe_call(img):
    try:
        t0 = time.time()
        resp = model.generate_content(
            [PROMPT, img],
            generation_config={"temperature": 0}
        )
        latency = time.time() - t0
        return resp, latency
    except google_exceptions.ResourceExhausted:
        print("[429] Rate limited")
        return None, None
    except Exception as e:
        print("[ERROR]", e)
        return None, None




# ============================================================
# 3. UNIVERSAL INFERENCE FUNCTION
# ============================================================
def run_inference(rows, tag):
    preds, truths, latencies = [], [], []
    out_path = BASE + f"{tag}_predictions_gemini_2.0_flash_light.jsonl"

    with open(out_path, "w") as out:
        print(f"\n===== Running Gemini on {tag.upper()} images =====")

        for idx, row in tqdm(rows.iterrows(), total=len(rows)):
            img = load_image(row["image_path"])
            if img is None:
                continue

            pred = None
            latency = None

            for attempt in range(3):
                resp, latency = safe_call(img)

                if resp is None:
                    time.sleep(8 * (attempt + 1))
                    continue

                pred = extract_json(resp.text)
                if pred is not None:
                    break

            if pred is None:
                print(f"[SKIP] {tag} {idx}")
                continue

            out.write(json.dumps({
                "image_path": row["image_path"],
                "json": pred,
                "truth": row.to_dict(),
                "latency": latency
            }) + "\n")

            preds.append(pred)
            truths.append(row)
            latencies.append(latency)

            time.sleep(6)  # REQUIRED for free tier

    return preds, truths, latencies



# ============================================================
# 4. RUN SIMPLE & PROCESSED
# ============================================================
simple_preds, simple_truths, simple_lat = run_inference(simple_df, "simple")
processed_preds, processed_truths, processed_lat = run_inference(processed_df, "processed")







Simple: 5
Processed: 5

===== Running Gemini on SIMPLE images =====


100%|██████████| 5/5 [00:38<00:00,  7.63s/it]



===== Running Gemini on PROCESSED images =====


100%|██████████| 5/5 [00:38<00:00,  7.73s/it]


- Evaluation

In [ ]:
# script with normalization for human sensible result

# ============================================================
# 0. SETUP — LOAD SAVED JSONL FILES ONLY
# ============================================================

BASE = "/content/drive/MyDrive/FoodBenchmarkDataset/"
# simple_file    = BASE + "simple_predictions.jsonl"
# processed_file = BASE + "processed_predictions.jsonl"

simple_file    = BASE + "simple_predictions_gemini_2.0_flash_light.jsonl"
processed_file = BASE + "processed_predictions_gemini_2.0_flash_light.jsonl"


# ============================================================
# 1. TEXT NORMALIZATION
# ============================================================
REMOVE_WORDS = {
    "fresh","organic","raw","cooked","ripe","unripe","sweet",
    "mixed","assorted","prepared","whole","cut","slice","sliced",
    "yellow","white","green","red","boiled","steamed","steam",
    "with","and","in","style"
}

def normalize_text(s):
    if not s:
        return ""
    s = s.lower().strip()
    s = "".join(c for c in unicodedata.normalize("NFD", s)
                if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\s+"," ",s).strip()
    tokens = [t for t in s.split() if t not in REMOVE_WORDS]
    return " ".join(tokens)

# ============================================================
# 2. SEMANTIC DISH NAME NORMALIZATION
# ============================================================
NAME_SYNONYMS = {
    "corn": ["sweetcorn", "sweet corn", "corn kernels"],
    "egg": ["boiled egg", "egg boiled", "hard boiled egg"],
    "fruit": ["fruit bowl", "mixed fruit", "fruit platter"],
    "vegetable": ["mixed vegetables", "veggie mix"],
    "salad": ["greens", "vegetable salad"],
    "curry": ["yellow curry", "vegetable curry", "chicken curry"],
    "noodles": ["noodle", "noodle bowl", "noodle dish"]
}

# build reverse lookup
NAME_MAP = {}
for root, forms in NAME_SYNONYMS.items():
    NAME_MAP[root] = root
    for f in forms:
        NAME_MAP[f] = root

def normalize_dish_name(s):
    s = normalize_text(s)
    # exact synonym mapping
    if s in NAME_MAP:
        return NAME_MAP[s]
    # check partial matches
    for key in NAME_MAP:
        if key in s or s in key:
            return key
    return s

def name_match(pred, true):
    p = normalize_dish_name(pred)
    t = normalize_dish_name(true)

    if p == t:
        return True
    if p in t or t in p:
        return True
    if len(set(p.split()) & set(t.split())) > 0:
        return True
    return False

# ============================================================
# 3. SEMANTIC COOKING METHOD NORMALIZATION
# ============================================================
C_METHODS = {
    "fry": ["fried", "frying", "deep fried", "pan fried", "air fried",
            "stir fry","stir-fried", "shallow fried"],
    "boil": ["boiled", "boiling", "blanched", "poached", "simmered", "braised"],
    "steam": ["steamed","steaming"],
    "bake": ["baked","baking","oven roasted","oven baked"],
    "grill": ["grilled","grilling","barbeque","bbq"],
    "raw": ["uncooked","fresh"]
}

C_MAP = {}
for root, forms in C_METHODS.items():
    C_MAP[root] = root
    for f in forms:
        C_MAP[f] = root

def normalize_cook_method(s):
    s = normalize_text(s)
    tokens = s.split()
    roots = []
    for t in tokens:
        if t in C_MAP:
            roots.append(C_MAP[t])
        else:
            roots.append(t)
    return " ".join(roots)

def cooking_match(pred, true):
    p = normalize_cook_method(pred)
    t = normalize_cook_method(true)

    if p == t:
        return True
    if p in t or t in p:
        return True
    return len(set(p.split()) & set(t.split())) > 0

# ============================================================
# 4. PORTION SIZE MATCHING (±20% tolerance)
# ============================================================
def parse_portion_list(v):
    if not v:
        return []
    if isinstance(v, list):
        vals = v
    else:
        try: vals = ast.literal_eval(str(v))
        except: return []

    out = []
    for item in vals:
        if ":" not in item:
            continue
        name, weight = item.split(":",1)
        name = normalize_text(name)
        try:
            grams = float(weight.replace("g","").strip())
        except:
            grams = None
        out.append((name,grams))
    return out

def portion_match(pred, true):
    p_list = parse_portion_list(pred)
    t_list = parse_portion_list(true)
    if not p_list or not t_list:
        return False

    for pn, pg in p_list:
        for tn, tg in t_list:
            if len(set(pn.split()) & set(tn.split())) == 0:
                continue

            if pg and tg:
                if abs(pg - tg) / max(tg, 1) <= 0.20:
                    return True

            if pg is None or tg is None:
                return True
    return False

# ============================================================
# 5. NUTRITION ACCURACY (±20% tolerance)
# ============================================================
def parse_nutrition(v):
    if not v: return {}
    if isinstance(v, dict): return v
    try: return ast.literal_eval(str(v))
    except: return {}

N_FIELDS = ["calories_kcal", "protein_g", "fat_g", "carbohydrate_g"]

def nutrition_match(pred, true):
    p = parse_nutrition(pred)
    t = parse_nutrition(true)
    if not p or not t:
        return False

    correct = 0
    for key in N_FIELDS:
        if key not in p or key not in t:
            continue

        try:
            pv = float(p[key])
            tv = float(t[key])
        except:
            continue

        if tv == 0:
            continue

        if abs(pv - tv) / abs(tv) <= 0.20:
            correct += 1

    # Return how many N_FIELDS are correctly inferred out of the four(len(N_FIELDS)).
    return correct/4


# ============================================================
# 6. LOAD JSONL FILES
# ============================================================
def load_jsonl(path):
    preds, truths, lat = [], [], []
    with open(path,"r") as f:
        for line in f:
            o = json.loads(line)
            preds.append(o["json"])
            truths.append(o["truth"])
            lat.append(o.get("latency",0))
    return preds, truths, lat

simple_preds, simple_truths, simple_lat = load_jsonl(simple_file)
processed_preds, processed_truths, processed_lat = load_jsonl(processed_file)

# ============================================================
# 7. ACCURACY METRICS
# ============================================================
simple_name_acc = np.mean([name_match(p["dish_name"], t["label"])
                           for p,t in zip(simple_preds, simple_truths)])

processed_name_acc = np.mean([name_match(p["dish_name"], t["label"])
                              for p,t in zip(processed_preds, processed_truths)])

cooking_acc = np.mean([cooking_match(p["cooking_method"], t["cooking_method"])
                       for p,t in zip(processed_preds, processed_truths)])

portion_acc = np.mean([portion_match(p["portion_size"], t["portion_size"])
                       for p,t in zip(processed_preds, processed_truths)])

nutrition_acc = np.mean([
    nutrition_match(p["nutritional_profile"], t["calories_kcal"] if False else t.get("calories_kcal"))
    for p,t in zip(processed_preds, processed_truths)
])

# ============================================================
# 8. RESULTS
# ============================================================
print("\n================ BENCHMARK RESULTS ================")

print("\n--- SIMPLE FOODS ---")
print("Samples:", len(simple_preds))
print("Dish Name Accuracy:", round(simple_name_acc,3))
print("Avg Latency:", round(np.mean(simple_lat),3))

print("\n--- PROCESSED FOODS ---")
print("Samples:", len(processed_preds))
print("Dish Name Accuracy:", round(processed_name_acc,3))
print("Cooking Method Accuracy:", round(cooking_acc,3))
print("Portion Size Accuracy:", round(portion_acc,3))
print("Nutrition Accuracy:", round(nutrition_acc,3))
print("Avg Latency:", round(np.mean(processed_lat),3))



================ BENCHMARK RESULTS ================

--- SIMPLE FOODS ---
Samples: 100
Dish Name Accuracy: 0.8
Avg Latency: 1.665

--- PROCESSED FOODS ---
Samples: 400
Dish Name Accuracy: 0.685
Cooking Method Accuracy: 0.785
Portion Size Accuracy: 0.418
Nutrition Accuracy: 0.488
Avg Latency: 1.975
